# 🎙️ MOSS-TTS-v1.5 (8B) Narrator

Paste a full narration script and get **one** downloadable audio file, using [OpenMOSS MOSS-TTS-v1.5](https://github.com/OpenMOSS/MOSS-TTS) (the 8B model).

**How to use**
1. **Runtime → Change runtime type → A100 GPU** (needs Colab Pro / Pay As You Go).
2. **Runtime → Run all**. The first run downloads ~24 GB of model weights (about 5 to 10 minutes).
3. Paste your text in the box that appears at the bottom and click **Generate narration**. The file downloads automatically when it's done.

Optional: tick **clone_voice** in step 3 before Run all to upload a 5 to 30 second voice sample.

In [ ]:
#@title 1️⃣ Setup: check GPU and install dependencies (about 2 min)
import os, sys, torch

if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. Runtime → Change runtime type → A100 GPU, then Runtime → Run all.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} ({gpu.total_memory / 1024**3:.0f} GB)")
if gpu.total_memory / 1024**3 < 30:
    raise SystemExit("❌ The 8B model needs an A100 (40 GB). Runtime → Change runtime type → A100 GPU.")

REPO_DIR = "/content/moss-tts-narrator"
if not os.path.isdir(REPO_DIR):
    !git clone -q --depth 1 https://github.com/GITHUB_USER/moss-tts-narrator.git {REPO_DIR}
!pip install -q -r {REPO_DIR}/requirements-colab.txt
import importlib.util
if importlib.util.find_spec("torchaudio") is None:  # the model's remote code imports it
    TORCH_VERSION = torch.__version__.split("+")[0]
    !pip install -q "torchaudio=={TORCH_VERSION}"
if "transformers" in sys.modules:
    print("⚠️ transformers was imported before the upgrade: Runtime → Restart session, then Run all again.")
sys.path.insert(0, REPO_DIR)
print("✅ Setup done")

In [ ]:
#@title 2️⃣ Load MOSS-TTS-v1.5 8B (first run downloads ~24 GB)
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
from moss_narrator.engine import MossNarrator

if "narrator" not in globals():
    narrator = MossNarrator.load()
print("✅ Model ready")

In [ ]:
#@title 3️⃣ (Optional) Clone a voice from a sample
clone_voice = False #@param {type:"boolean"}
#@markdown Tick, then run this cell to upload 5 to 30 s of clean speech (wav / mp3 / m4a). Leave unticked to use the model's own voice (kept consistent across the whole narration).
import os, shutil

REFERENCE_PATH = None
if clone_voice:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        os.makedirs("/content/voice_samples", exist_ok=True)
        name = next(iter(uploaded))
        REFERENCE_PATH = shutil.move(name, os.path.join("/content/voice_samples", os.path.basename(name)))
        print("🎤 Using voice sample:", REFERENCE_PATH)

In [ ]:
#@title 4️⃣ Paste your narration, then click **Generate narration**
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
from moss_narrator import colab_ui

colab_ui.launch(narrator, reference_path=REFERENCE_PATH)